# NIH Tracker maintenance notebook (v19.2.1; extends v19.2)

This production-only notebook contains the low-frequency operational controls intentionally excluded from the pipeline and review notebooks: release installation or refresh, reference updates, lock handling, bounded cleanup, transaction restoration, and archive-backed reset.

Destructive operations are preview-first and require checkboxes. Exact CLI confirmation tokens are generated internally rather than entered into free-text fields.


## 1. Install or refresh the production release

Run this block only when installing v19.2.1 or repairing release code. It updates only `/content/drive/MyDrive/NIH_Tracker`; it does not scan or recreate dev/staging. Release-managed scripts, notebooks, references, templates, and documentation may be updated, while cards, inputs, runs, snapshots, builds, locks, caches, and credentials are preserved.

**Create missing files** never overwrites an existing release file. **Refresh release code** replaces only changed release-managed files.


In [ ]:
PREFILL_MODE = "Refresh release code"  # @param ["Refresh release code", "Create missing files"]
APPLY_PREFILL = False  # @param {type:"boolean"}

import datetime as dt
import hashlib
import os
import shutil
import subprocess
import sys
import uuid
import zipfile
from pathlib import Path

PRODUCTION_ROOT = Path("/content/drive/MyDrive/NIH_Tracker")
try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except Exception:
    pass

os.environ["NIH_TRACKER_PRODUCTION_ONLY"] = "1"
os.environ["NIH_TRACKER_ROOT_PROD"] = str(PRODUCTION_ROOT)
os.environ["NIH_TRACKER_ENV"] = "prod"
os.environ["NIH_TRACKER_SHARED_CACHE_ROOT"] = "/content/drive/MyDrive/NIH_Tracker_shared/cache"
os.environ.pop("NIH_TRACKER", None)
os.environ.pop("NIH_TRACKER_ROOT_DEV", None)
os.environ.pop("NIH_TRACKER_ROOT_STAGING", None)
os.environ.pop("NIH_TRACKER_ALLOW_PROJECT_OVERRIDE", None)


def _sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _validate_release_inventory(release_root):
    sums_path = release_root / "SHA256SUMS"
    if not sums_path.is_file():
        raise FileNotFoundError(f"Missing release inventory: {sums_path}")
    checked = 0
    for raw_line in sums_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        expected, relative_text = line.split(maxsplit=1)
        relative = Path(relative_text.lstrip("*").strip())
        if relative.is_absolute() or ".." in relative.parts:
            raise RuntimeError(f"Unsafe SHA256SUMS path: {relative}")
        target = (release_root / relative).resolve()
        target.relative_to(release_root.resolve())
        if not target.is_file() or _sha256(target) != expected:
            raise RuntimeError(f"Release inventory mismatch: {relative}")
        checked += 1
    if not checked:
        raise RuntimeError("SHA256SUMS contains no verifiable files.")
    print(f"Verified {checked} release file hash(es) from {sums_path}.")


def _safe_extract_release(bundle):
    bundle = bundle.resolve()
    bundle_sha = _sha256(bundle)
    extraction_parent = Path("/content/nih_tracker_v19_bootstrap")
    extraction_parent.mkdir(parents=True, exist_ok=True)
    destination = extraction_parent / bundle_sha[:16]
    marker = destination / ".bundle_sha256"
    if marker.is_file() and marker.read_text(encoding="utf-8").strip() == bundle_sha:
        return destination
    temporary = extraction_parent / f".extract-{uuid.uuid4().hex}"
    temporary.mkdir(parents=True, exist_ok=False)
    total_size = 0
    file_count = 0
    try:
        with zipfile.ZipFile(bundle) as archive:
            for member in archive.infolist():
                relative = Path(member.filename)
                if relative.is_absolute() or ".." in relative.parts:
                    raise RuntimeError(f"Unsafe ZIP member: {member.filename!r}")
                unix_type = (member.external_attr >> 16) & 0o170000
                if unix_type == 0o120000:
                    raise RuntimeError(f"Symlink ZIP member is not allowed: {member.filename!r}")
                target = (temporary / relative).resolve()
                target.relative_to(temporary.resolve())
                if member.is_dir():
                    target.mkdir(parents=True, exist_ok=True)
                    continue
                file_count += 1
                total_size += int(member.file_size)
                if file_count > 5000 or total_size > 500 * 1024 * 1024:
                    raise RuntimeError("Release ZIP exceeds the safe extraction limit.")
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, target.open("wb") as output:
                    shutil.copyfileobj(source, output)
        if destination.exists():
            shutil.rmtree(destination)
        os.replace(temporary, destination)
        marker.write_text(bundle_sha + "\n", encoding="utf-8")
    except Exception:
        shutil.rmtree(temporary, ignore_errors=True)
        raise
    print(f"Safely extracted release bundle: {bundle} -> {destination}")
    return destination


bundle_candidates = []
for parent in (Path("/content"), Path("/content/drive/MyDrive")):
    if parent.is_dir():
        bundle_candidates.extend(parent.glob("nih_tracker_v19_2_1_bundle.zip"))
newest_bundle = next(
    iter(sorted(
        {path.resolve() for path in bundle_candidates if path.is_file()},
        key=lambda path: path.stat().st_mtime_ns,
        reverse=True,
    )),
    None,
)
release_roots = []
if newest_bundle:
    release_roots.append(_safe_extract_release(newest_bundle))
release_roots.extend([
    Path("/content/NIH_Tracker_release_v19"),
    Path("/content/drive/MyDrive/NIH_Tracker_release_v19"),
    Path.cwd(),
])
script_candidates = []
for release_root in release_roots:
    release_root = release_root.expanduser().resolve()
    direct = release_root / "scripts" / "step_initialize_environments.py"
    if direct.is_file():
        script_candidates.append(direct)
    if release_root.is_dir():
        script_candidates.extend(release_root.glob("*/scripts/step_initialize_environments.py"))
script_candidates.append(PRODUCTION_ROOT / "scripts" / "step_initialize_environments.py")
INITIALIZER = next((path.resolve() for path in script_candidates if path.is_file()), None)
if INITIALIZER is None:
    raise FileNotFoundError(
        "Production release installer was not found. Upload/extract the v19.2.1 bundle to /content."
    )
RELEASE_ROOT = INITIALIZER.parents[1]
_validate_release_inventory(RELEASE_ROOT)
import json
EXPECTED_RELEASE_VERSION = "19.2.1"
installed_manifest = json.loads((RELEASE_ROOT / "RELEASE_MANIFEST.json").read_text())
if installed_manifest.get("release") != EXPECTED_RELEASE_VERSION:
    raise RuntimeError("Upload nih_tracker_v19_2_1_bundle.zip to /content; an older release cannot be installed by this notebook.")

mode = "missing-only" if PREFILL_MODE == "Create missing files" else "refresh-code"
command = [
    sys.executable,
    str(INITIALIZER),
    "--source-env", "prod",
    "--mode", mode,
    "--release-root", str(RELEASE_ROOT),
]
if APPLY_PREFILL:
    command.extend(["--apply", "--confirm", "INSTALL_PRODUCTION_RELEASE"])
log_dir = PRODUCTION_ROOT / "runs" / "bootstrap_notebook"
log_dir.mkdir(parents=True, exist_ok=True)
stamp = dt.datetime.now().strftime("%Y%m%dT%H%M%S%f")
log_path = log_dir / f"{stamp}.log"
result_path = log_dir / f"{stamp}.result.json"
print(f"Installer: {INITIALIZER}")
print(f"Release root: {RELEASE_ROOT}")
print(f"Debug log: {log_path}")
print("$ " + " ".join(command))
started = dt.datetime.now(dt.timezone.utc)
with log_path.open("w", encoding="utf-8") as handle:
    process = subprocess.Popen(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
        bufsize=1,
    )
    assert process.stdout is not None
    with process.stdout:
        for line in process.stdout:
            print(line, end="")
            handle.write(line)
            handle.flush()
    return_code = process.wait()
finished = dt.datetime.now(dt.timezone.utc)
result_path.write_text(
    __import__("json").dumps(
        {
            "schema": "nih-tracker-bootstrap-notebook-result/v2",
            "captured_at": finished.isoformat(timespec="seconds"),
            "environment": "prod",
            "mode": mode,
            "apply": APPLY_PREFILL,
            "initializer": str(INITIALIZER),
            "release_root": str(RELEASE_ROOT),
            "command": command,
            "returncode": return_code,
            "duration_seconds": round((finished - started).total_seconds(), 3),
            "log": str(log_path),
        },
        indent=2,
    ) + "\n",
    encoding="utf-8",
)
print(f"Return code: {return_code}")
print(f"Result record: {result_path}")
if return_code != 0:
    raise RuntimeError(f"Production release installation failed; inspect {log_path}")


## 2. Start production maintenance

After the release block has installed the scripts, start the production runtime. Card restoration, reset, and reference updates require the production-write acknowledgement where indicated.


In [ ]:
NOTEBOOK_NAME = "maintenance"

ENVIRONMENT = "prod"
ENABLE_PROD_WRITES = False  # @param {type:"boolean"}

import os
import sys
from pathlib import Path

ROOTS = {"prod": "/content/drive/MyDrive/NIH_Tracker"}
os.environ["NIH_TRACKER_PRODUCTION_ONLY"] = "1"
os.environ["NIH_TRACKER_ROOT_PROD"] = ROOTS["prod"]
os.environ["NIH_TRACKER_ENV"] = "prod"
os.environ["NIH_TRACKER_SHARED_CACHE_ROOT"] = "/content/drive/MyDrive/NIH_Tracker_shared/cache"
os.environ.pop("NIH_TRACKER", None)
os.environ.pop("NIH_TRACKER_ALLOW_PROJECT_OVERRIDE", None)
os.environ.pop("NIH_TRACKER_ROOT_DEV", None)
os.environ.pop("NIH_TRACKER_ROOT_STAGING", None)

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except Exception:
    pass

SCRIPTS = Path(ROOTS["prod"]) / "scripts"
if not SCRIPTS.is_dir():
    raise FileNotFoundError(
        f"Missing {SCRIPTS}. Run the maintenance notebook's production release installer first."
    )
previous_scripts = globals().get("_NIH_TRACKER_PREVIOUS_SCRIPTS")
if previous_scripts and previous_scripts in sys.path:
    sys.path.remove(previous_scripts)
if str(SCRIPTS) in sys.path:
    sys.path.remove(str(SCRIPTS))
sys.path.insert(0, str(SCRIPTS))
_NIH_TRACKER_PREVIOUS_SCRIPTS = str(SCRIPTS)
for module_name in list(sys.modules):
    if (
        module_name in {"tracker_gcp_backend", "evidence_store", "evidence_collector", "capability_assessment", "gcp_review", "gcp_workbench", "gcp_config", "tracker_cache_bridge", "gcp_review_queue", "tracker_claude_backend", "maintenance_widgets", "run_provenance", "code_provenance", "step_project_inventory", "common", "identity_hygiene", "interoperability_schema", "resource_profile_schema", "policy_context_schema", "repository_status_schema", "publisher_context_schema", "notebook_runtime", "review_widgets", "review_workflow", "automated_review", "verify_external_ids", "catalog_schema", "catalog_vocab", "catalog_claims", "catalog_graph", "catalog_resolver", "catalog_history", "catalog_exports"}
        or module_name.startswith("step_")
    ):
        sys.modules.pop(module_name, None)

from notebook_runtime import NotebookRuntime
RUNTIME = NotebookRuntime.create(
    notebook_name=NOTEBOOK_NAME,
    env="prod",
    enable_prod_writes=ENABLE_PROD_WRITES,
    roots=ROOTS,
)
common = RUNTIME.common
print(f"Ready: notebook={NOTEBOOK_NAME} | production | project={common.PROJECT}")
print(f"Debug session: {RUNTIME.debug_dir}")


## 3. Inspect storage, locks, and debug state

This records a structured runtime diagnostic and a maintenance status report before any destructive operation.


In [ ]:
DIAGNOSTICS = RUNTIME.show_diagnostics()
status_result = RUNTIME.run_script(
    "step_maintenance.py",
    ["status"],
    label="maintenance_status",
)
if not status_result.ok:
    raise RuntimeError(f"Maintenance status failed; inspect {status_result.log_path}")


## 3a. Inventory and pin review sources

Run **Inventory only** first. Stop other notebooks and background writers before
capture. The report hashes existing files, identifies selected review sources,
and records protected dependencies. It does not move or remove project files.
Open the generated `index.html` locally; it contains internal operational paths
and must not be added to the public site.

After inspecting the selected source paths and SHA-256 values, choose **Pin
current review sources** to keep those exact queues selected during review.
Pins do not approve candidates. Changed or missing pinned bytes stop the review
dashboard. **Unpin review sources** preserves the prior selection in the audit
history; use it before intentionally selecting new runs after a pipeline refresh.


In [ ]:
# NIH_TRACKER_OPERATIONS_V19_2
PROJECT_ACTION = "Inventory only" # @param ["Inventory only", "Pin current review sources", "Unpin review sources"]
CONFIRM_SOURCE_SELECTION_CHANGE = False # @param {type:"boolean"}
if PROJECT_ACTION == "Inventory only":
    inventory_result = RUNTIME.run_script("step_project_inventory.py", ["inventory"], label="project_inventory")
    if not inventory_result.ok:
        print("Inventory is incomplete. Inspect its report and log before proceeding.")
elif CONFIRM_SOURCE_SELECTION_CHANGE:
    action = "pin-review" if PROJECT_ACTION.startswith("Pin") else "unpin-review"
    token = "PIN_REVIEW_SOURCES" if action == "pin-review" else "UNPIN_REVIEW_SOURCES"
    selection_result = RUNTIME.run_script("step_project_inventory.py", [action, "--confirm", token], label="review_source_selection")
    if not selection_result.ok:
        raise RuntimeError("Review source selection failed; inspect the log.")
else:
    print("Inspect the inventory, then acknowledge the selected source change.")


## 4. Refresh BMIC, dbGaP, or publisher-policy source information

Select one bounded update. For BMIC, upload a CSV whose filename contains `BMIC` to `/content`; when no upload is present, the updater validates the existing environment snapshot and reports a no-op. The dbGaP updater follows the official FHIR `ResearchStudy` catalogue and honors the response charset recorded by the service. Publisher refresh permits a bounded partial update: successfully fetched pages are pinned, while up to two transiently failed source rows retain their prior fields unchanged and are listed for retry. No updater changes semantic publisher-to-repository assertions automatically.

Run once without **Apply** to inspect the structured report. Enable **Apply** only after reviewing the candidate, failed URLs, row counts, and source hashes.


In [ ]:
REFERENCE_UPDATE = "No update"  # @param ["No update", "Validate/update BMIC snapshot", "Refresh dbGaP FHIR snapshot", "Refresh publisher policy source snapshots"]
APPLY_REFERENCE_UPDATE = False  # @param {type:"boolean"}

from pathlib import Path
import json
import pandas as pd
from IPython.display import display

if REFERENCE_UPDATE == "No update":
    print("No reference update selected.")
else:
    if APPLY_REFERENCE_UPDATE and common.ENV == "prod" and not ENABLE_PROD_WRITES:
        raise RuntimeError("Production reference updates require ENABLE_PROD_WRITES in setup.")
    if REFERENCE_UPDATE == "Validate/update BMIC snapshot":
        script = "step_update_bmic_info.py"
        arguments = []  # auto-detect /content upload; fall back to current environment snapshot
        label = "update_bmic_info"
        step_folder = "update_bmic_info"
        report_files = ["update_report.json", "update_error.json"]
    elif REFERENCE_UPDATE == "Refresh dbGaP FHIR snapshot":
        script = "step_update_dbgap_info.py"
        arguments = ["--mode", "fhir"]
        label = "update_dbgap_info"
        step_folder = "update_dbgap_info"
        report_files = ["update_report.json", "update_error.json"]
    else:
        script = "step_update_publisher_info.py"
        arguments = ["--continue-on-error", "--max-fetch-errors", "2"]
        label = "update_publisher_info"
        step_folder = "update_publisher_info"
        report_files = [
            "coverage.json",
            "publisher_source_drift_report.csv",
            "publisher_policy_sources.proposed.csv",
        ]
    if APPLY_REFERENCE_UPDATE:
        arguments.append("--apply")
    update_result = RUNTIME.run_script(
        script,
        arguments,
        label=label,
        writes_cards=bool(APPLY_REFERENCE_UPDATE),
    )
    print(
        ("Reference update completed." if update_result.ok else "Reference update stopped safely.")
        + f" Debug log: {update_result.log_path}"
    )

    step_root = Path(common.RUNS) / step_folder
    run_dirs = sorted(
        [path for path in step_root.iterdir() if path.is_dir()],
        key=lambda path: path.stat().st_mtime_ns,
        reverse=True,
    ) if step_root.is_dir() else []
    if run_dirs:
        latest_update_dir = run_dirs[0]
        print(f"Structured update directory: {latest_update_dir}")
        for report_name in report_files:
            report_path = latest_update_dir / report_name
            if not report_path.is_file():
                continue
            print()
            print(f"--- {report_name} ---")
            if report_path.suffix.casefold() == ".json":
                print(json.dumps(json.loads(report_path.read_text(encoding="utf-8")), indent=2))
            else:
                report_frame = pd.read_csv(report_path, dtype=str, keep_default_na=False)
                display(report_frame.head(100))


## 5. Inspect or remove an abandoned card lock

Lock age alone is not proof that a writer has stopped. Removal is permitted only after independently confirming that no notebook or process is writing cards.


In [ ]:
LOCK_ACTION = "Inspect only"  # @param ["Inspect only", "Remove confirmed abandoned lock"]
CONFIRM_NO_ACTIVE_WRITER = False  # @param {type:"boolean"}

arguments = ["remove-lock"]
if LOCK_ACTION == "Remove confirmed abandoned lock":
    if not CONFIRM_NO_ACTIVE_WRITER:
        raise RuntimeError(
            "Check CONFIRM_NO_ACTIVE_WRITER only after independently confirming all writers stopped."
        )
    arguments.extend(["--apply", "--confirm", "REMOVE_ABANDONED_CARD_LOCK"])
lock_result = RUNTIME.run_script(
    "step_maintenance.py",
    arguments,
    label="card_lock_maintenance",
)
if not lock_result.ok:
    raise RuntimeError(f"Lock maintenance failed; inspect {lock_result.log_path}")


## 6. Preview or apply bounded cleanup

Run the cell to list saved, unapplied sealed plans. No plan is selected by default.
Choose a plan and click **Inspect selected plan** to see its exact path, SHA-256,
eligible paths and protected paths. If the list is empty, click **Create plan**.
Apply is enabled only after inspection and both acknowledgments. Changed plan
bytes require fresh inspection; candidate changes are checked before deletion.

Structured run outputs,
decision dependencies and card transaction history are retained even beyond the
age/count threshold. Eligible logs and temporary paths are archived and every
archived file hash is verified before removal. Recovery ZIPs and receipts are
retained under `snapshots/maintenance/` and the plan folder. A small or empty
deletion set is expected when the ledger references historical runs.


In [ ]:
# NIH_TRACKER_CLEANUP_CONTROLS_V19_2_1
KEEP_RUNS_PER_STEP = 5 # @param [3, 5, 10, 20]
REMOVE_TEMPORARY_FILES = False # @param {type:"boolean"}
from maintenance_widgets import launch_cleanup_controls
cleanup_controls = launch_cleanup_controls(
    RUNTIME, keep_runs=KEEP_RUNS_PER_STEP, remove_temporary=REMOVE_TEMPORARY_FILES
)


## 7. Restore a committed card transaction

Select a committed transaction from the dropdown. Preview verifies that current files still match the recorded after-images. Applying restoration creates a new transaction and will not overwrite legitimate later changes.


In [ ]:
import json
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

manifest_options = []
for manifest_path in sorted(
    (Path(common.SNAPSHOTS) / "cards").glob("*/manifest.json"),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
):
    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except Exception:
        continue
    if manifest.get("status") != "committed":
        continue
    label = (
        f"{manifest.get('completed_at') or manifest.get('started_at')} | "
        f"{manifest.get('step')} | {manifest.get('transaction_id')} | "
        f"{len(manifest.get('touched', []))} file(s)"
    )
    manifest_options.append((label, str(manifest_path)))

manifest_widget = widgets.Dropdown(
    options=manifest_options,
    description="Transaction",
    layout=widgets.Layout(width="1100px"),
)
apply_restore_widget = widgets.Checkbox(
    value=False,
    description="Apply verified restoration",
    indent=False,
)
restore_button = widgets.Button(
    description="Preview/restore selected transaction",
    button_style="warning",
    icon="undo",
    disabled=not bool(manifest_options),
)
restore_output = widgets.Output()


def _restore(_button):
    with restore_output:
        clear_output(wait=True)
        if not manifest_widget.value:
            print("No committed transaction is available.")
            return
        manifest_path = Path(manifest_widget.value)
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        arguments = ["--manifest", str(manifest_path)]
        writes_cards = bool(apply_restore_widget.value)
        if writes_cards:
            if common.ENV == "prod" and not ENABLE_PROD_WRITES:
                print("Production restoration requires ENABLE_PROD_WRITES in setup.")
                return
            arguments.extend(
                [
                    "--apply",
                    "--confirm",
                    f"RESTORE_{manifest['transaction_id']}",
                ]
            )
        result = RUNTIME.run_script(
            "step_restore_card_snapshot.py",
            arguments,
            label="restore_card_transaction",
            writes_cards=writes_cards,
        )
        print(f"Result: {'PASS' if result.ok else 'FAIL'} | {result.log_path}")

restore_button.on_click(_restore)
display(
    widgets.VBox(
        [manifest_widget, apply_restore_widget, restore_button, restore_output]
    )
)


## 8. Archive and reset all cards

Reset first creates a collision-resistant ZIP archive, then deletes card files inside one rollback-capable transaction. Select the reset action and check the confirmation box only when a full environment rebuild is intended.


In [ ]:
RESET_ACTION = "Do not reset"  # @param ["Do not reset", "Archive and reset all cards"]
CONFIRM_RESET = False  # @param {type:"boolean"}

reset_arguments = ["reset-cards"]
writes_cards = False
if RESET_ACTION == "Archive and reset all cards":
    if not CONFIRM_RESET:
        print("Reset preview only. Check CONFIRM_RESET to perform the archived reset.")
    else:
        if common.ENV == "prod" and not ENABLE_PROD_WRITES:
            raise RuntimeError(
                "Production reset requires ENABLE_PROD_WRITES in setup."
            )
        reset_arguments.extend(
            [
                "--apply",
                "--confirm",
                f"ARCHIVE_AND_RESET_{common.ENV.upper()}_CARDS",
            ]
        )
        writes_cards = True
reset_result = RUNTIME.run_script(
    "step_maintenance.py",
    reset_arguments,
    label="archive_and_reset_cards",
    writes_cards=writes_cards,
)
if not reset_result.ok:
    raise RuntimeError(f"Reset operation failed; inspect {reset_result.log_path}")
RUNTIME.write_session_summary()


## 9. Persist the maintenance audit summary

This final read-only cell consolidates operation results, recent ledger entries, and the current diagnostic state into the notebook debug directory.


In [ ]:
import pandas as pd
from IPython.display import display

RUNTIME.show_diagnostics()
if RUNTIME.run_results():
    display(pd.DataFrame(RUNTIME.run_results()))
if RUNTIME.recent_ledger(20):
    display(pd.DataFrame(RUNTIME.recent_ledger(20)))
RUNTIME.write_session_summary()


## v19.1 GCP extension notes

Install this v19.1.1 package using the existing production release installer.
The new `requirements-gcp.txt`, source-cache modules, assessment rubric and review
panel are release-managed; existing cards, inputs, snapshots, caches and reviews
are preserved. Use the review workbench's optional GCP panel for execution.
The catalog build remains offline. No cloud resource or recurring job is created
by this notebook automatically. See `GCP_INTEGRATION_AND_ASSESSMENT.md`.
